#  What is RAG? (And Why Do We Need It?)

Before we dive into Vectors and Databases, let's establish our ultimate goal: building a **RAG** system.

**RAG** stands for **Retrieval-Augmented Generation**. Let's break down exactly what that means:
- **Retrieval (R)**: The system first *searches* and *retrieves* relevant facts from an external knowledge base or database based on the user's question.
- **Augmented (A)**: The user's original prompt is then *augmented* (enriched) by securely attaching those retrieved facts to it.
- **Generation (G)**: Finally, a Large Language Model (LLM) reads the augmented prompt and *generates* an accurate, context-aware answer.

Essentially, it is an AI framework that improves the quality, accuracy, and reliability of Generative AI models by fetching facts before answering.

###  The Motivation:
Standard Large Language Models (LLMs) have incredible linguistic capabilities, but they face a few fundamental limitations:
1. **Knowledge Cutoff**: Their training data is frozen in time at the moment they were trained, meaning they lack awareness of current events or newer information.
2. **Context Window Sizes**: LLMs can only take in a limited amount of text at once. You cannot simply paste an entire enterprise database or thousands of pages into the prompt.
3. **Private Datasets**: Standard models do not have access to your proprietary logic, personal files, and internal company documents.

###  The RAG Solution:
RAG fundamentally addresses these limitations without needing to continuously retrain the LLM:
1. **Versatile Updates for Knowledge**: RAG allows you to update the external database dynamically, ensuring the model always answers from the freshest, most up-to-date information.
2. **Better Managing the Context Windows**: Instead of loading huge documents all at once, RAG precisely searches and extracts only the highly relevant chunks of text needed to answer the question, efficiently utilizing the context window.
3. **Integration Feature for Private Databases**: RAG connects LLMs directly to your existing private databases, keeping your proprietary data secure while enabling the AI to reason securely over internal documents.

##  Components of a RAG System

A standard RAG pipeline consists of two primary components working seamlessly together:

1. **The Retriever**: This acts as the intelligent search engine. When a user asks a question, the retriever scans the external knowledge base to find the most relevant document chunks. To do this efficiently with text, it relies heavily on an **Embedding Model** and a **Vector Database**.
2. **The Generator**: This is the Large Language Model (LLM). It takes the user's original question *plus* the verified context found by the Retriever, and synthesizes a clear, human-like response.

Since the Retriever forms the foundation of the "Search" capability, we first need to understand how it organizes and looks up information. That brings us to **Vector Databases**!

#  Introduction to Vector Databases
Welcome to this quick primer on Vector Databases! Before we build our full Retrieval-Augmented Generation (RAG) system, we need to understand how the "Retrieval" part actually works. 

In this notebook, we covers:
1. **Setup & Initialization**: Loading our tools (`numpy`, `sentence-transformers`, `qdrant-client`).
2. **Text Embeddings**: Converting text to numbers.
3. **Local Vector Database**: Creating an in-memory database to store our numbers.
4. **Inserting Data**: Putting vectors in our DB.
5. **Similarity Search**: Finding the "closest" vectors using Cosine Similarity!

## 1. Setup and Library Initialization

First, let's install and import the required libraries:
- `sentence-transformers`: To generate text embeddings.
- `qdrant-client`: Represents our local vector database here.
- `numpy`: To assist with some math functions.

In [ ]:
# Install necessary libraries (uncomment if running outside Colab and missing these)
!pip install sentence-transformers qdrant-client numpy

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

print("Libraries imported successfully!")

## 2. Generate Text Embeddings

A basic requirement for a Vector Database is having **vectors**. Text models convert raw text into thick numerical arrays (dense vectors), which encapsulate meaning and semantic relationships. We call these "Text Embeddings".

In [ ]:
# 1. Initialize the embedding model
encoder = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Define a dataset of 10 text chunks about RAG, each touching a distinct concept
documents = [
    "Definition: RAG (Retrieval-Augmented Generation) is an AI framework that enriches user prompts with facts fetched from an external database before generating an answer.",
    "Knowledge Cutoff Solution: RAG overcomes the 'frozen in time' limitation of standard LLMs by fetching the freshest information from dynamically updated databases.",
    "Context Window Optimization: Instead of loading huge documents all at once, RAG precisely extracts only the highly relevant textual chunks to fit within the LLM's context window.",
    "Data Security & Privacy: RAG securely connects language models directly to private enterprise databases, keeping proprietary logic and internal files completely secure.",
    "The Retriever Component: Acting as an intelligent search engine, the retriever scans the knowledge base using embedding models and vector databases to find relevant chunks.",
    "The Generator Component: The Large Language Model (Generator) takes the user's original question plus the verified context found by the Retriever to synthesize a response.",
    "No Continuous Retraining: RAG improves the quality, accuracy, and reliability of Generative AI without the need to continuously retrain the underlying LLM.",
    "Text Embeddings: In the retrieval phase, raw text is converted into thick numerical arrays (dense vectors) that deeply encapsulate meaning and semantic relationships.",
    "Vector Databases: These databases are built to store dense vectors and match them efficiently against incoming queries using mathematical similarity metrics.",
    "Cosine Similarity Search: RAG systems frequently use cosine similarity to measure the angle between vectors, finding database entries that semantically match the user's question."
]

# 3. Generate embeddings
embeddings = encoder.encode(documents)

print(f"Generated {len(embeddings)} embeddings.")
print(f"Each embedding has a dimension size of: {len(embeddings[0])}")
print(f"Sample of the first embedding's values: {embeddings[0][:5]} ...")

## 3. Initialize a Local Vector Database

We need a place to put these embeddings so we can search through them efficiently. We'll use Qdrant here. For learning purposes, we can run Qdrant uniquely in memory (meaning it vanishes when the notebook is closed) rather than connecting to a cloud node.

In [ ]:
# Initialize an in-memory db client
client = QdrantClient(":memory:")

# Vector DBs organize data in "collections" (like tables in a relational DB)
# We must specify the size (dimensions) of vectors going into this collection
# We also specify the distance metric we'll use to compare vector arrays. Cosine is standard for text!
collection_name = "demo_collection"

client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=encoder.get_sentence_embedding_dimension(), # Will be 384 for all-MiniLM-L6-v2
        distance=Distance.COSINE
    )
)

print(f"Collection '{collection_name}' has been created successfully!")

## 4. Insert Vectors and Metadata

We upload the numerical vector representations, but we *also* attach the original uncompressed text as `payload` (metadata). When the DB matches the vector, we can read the payload to figure out what text chunk it matched.

In [ ]:
# We assemble 'PointStruct' objects. Each point corresponds to a single vector and its metadata
points = [
    PointStruct(
        id=idx, 
        vector=embedding.tolist(), 
        payload={"text": doc} # This is the metadata!
    )
    for idx, (doc, embedding) in enumerate(zip(documents, embeddings))
]

# Upsert (Upload/Insert) to the Qdrant DB
operation_info = client.upsert(
    collection_name=collection_name,
    points=points
)

print("Insertion completed!", operation_info)

## 5. Perform Vector Similarity Search

Now for the magic! How does semantic search actually work? 
When a user asks a question, we run their specific text through the **exact same** embedding model to turn their query into a vector. Then, we ask the database to find the vectors stored inside it that are mathematically closest to the question vector. 

Since we set `Distance.COSINE` when building the collection, Qdrant will calculate the cosine similarity between the query vector and all target vectors using this formula:
$$ \cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|} $$

Where $\mathbf{A}$ is our question query vector, and $\mathbf{B}$ is a vector in the database.

A Cosine Similarity near 1 means highly similar context! Near 0 means completely orthoganol (unrelated) concepts. Negative means functionally opposite. Let's try it out!

In [ ]:
query_text = "What is Retrieval Augmented Generation used for?"
print(f"🔍 Searching for: '{query_text}'\n")

# Need to embed the query using the SAME model!
query_vector = encoder.encode(query_text).tolist()

search_results = client.query_points(
    collection_name=collection_name,
    query=query_vector, # query parameter is commonly used in query_points
    limit=2 # We only want the top 2 closest matches
).points

for result in search_results:
    # Notice the similarity score generated under `result.score`
    print(f"Score: {result.score:.4f} | Document: {result.payload['text']}")

## 6. Beyond the Basics: Embeddings, Indexing, and Search Methods

While we used a simple dense model and brute-force search in our toy example, real-world Vector Databases utilize advanced techniques to handle millions of vectors efficiently. Here is a quick overview of the different methods used in the industry:

### 🧠 1. Types of Embeddings
Not all vector representations are created equal!
- **Sparse Embeddings (Keyword-based)**: Algorithms like **TF-IDF** or **BM25**. These create massive, high-dimensional vectors mostly filled with zeros, where each dimension represents a specific word in the vocabulary. They are excellent for exact-keyword matching but fail at understanding nuanced context.
- **Dense Embeddings (Semantic-based)**: These create shorter, condensed vectors (e.g., 384 or 1536 dimensions) that capture the *underlying meaning* of the text, enabling matching even if the exact words differ. Common providers and models include:
  - **OpenAI's Embedding Models**:
    - `text-embedding-ada-002`
    - `text-embedding-davinci-001`
    - `text-embedding-curie-001`
    - `text-embedding-babbage-001`
    - `text-embedding-ada-001`
  - **Sentence Transformer Embeddings**:
    - `all-MiniLM-L6-v2`
    - `all-MiniLM-L12-v1`
    - `all-mpnet-base-v1`
    - `all-roberta-large-v1`
  - **Other models and providers**:
    - Google Vertex AI
    - Google PaLM
    - Aleph Alpha
    - Elasticsearch
    - ...

### 🗂️ 2. Indexing Algorithms
If you have millions of vectors, comparing your query to *every single one* (Brute-force) takes too long. Vector DBs use indexing algorithms to bypass this:
- **Flat Index (Brute-Force)**: Perfect accuracy, but slow. Checks every vector. (What we effectively did on our tiny dataset).
- **HNSW (Hierarchical Navigable Small World)**: The core algorithm for modern Vector DBs (including Qdrant defaults). It builds a multi-layered graph to quickly "hop" toward the closest vectors without checking everything. Extremely fast and highly accurate.
- **IVF (Inverted File Index)**: Groups similar vectors into clusters (Voronoi cells). During a search, the DB only checks the vectors inside the clusters that are physically closest to the query.
- **PQ (Product Quantization)**: A harsh compression technique that chunks and shrinks vectors to save massive amounts of RAM, trading off a bit of search accuracy.

### 🔍 3. Search and Distance Metrics
When the system tries to find the "nearest neighbors" to your query, how does it physically measure "distance"?
- **Cosine Similarity**: Measures the angle between two vectors. It is the gold standard for text because it ignores the pure magnitude (length of the document) and focuses purely on the direction (the semantic topic).
- **Dot Product**: Multiplies the vectors together. It is often preferred over cosine when vectors are already normalized (length of 1), as it is mathematically faster for computers to calculate.
- **Euclidean Distance (L2)**: Measures the actual straight-line distance between points. More common in computer vision, image processing, or tabular data.
- **KNN vs ANN**: K-Nearest Neighbors (KNN) guarantees the absolute best match but doesn't scale. Therefore, Vector DBs use Approximate Nearest Neighbors (ANN) to trade a tiny, unnoticeable fraction of accuracy for massive, game-changing gains in search speed.